# 05 — Evaluation, Threshold Optimization & Explainability

The winning pipeline was refit on train, its **decision threshold chosen on validation** to minimize
business cost — `cost = Σ(amount of missed frauds) + $5 × false alarms` — then evaluated **once** on
the held-out test set.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display
from src.utils import load_config, resolve_path

config = load_config()
pd.set_option("display.max_columns", 40)

In [ ]:
from src.utils import load_json
meta = load_json(resolve_path(config["paths"]["artifacts_dir"]) / "model_metadata.json")
ev = load_json(resolve_path(config["paths"]["artifacts_dir"]) / "test_evaluation.json")
print(f"Champion: {meta['model']} + {meta['strategy']} | tuned threshold = {meta['threshold']:.3f}")
pd.DataFrame({"default 0.50": ev["default_threshold"], f"tuned {meta['threshold']:.3f}": ev["tuned_threshold"]})

## Threshold: why not 0.5?

0.5 is only "neutral" if false positives and false negatives cost the same — here they differ by
orders of magnitude. The cost curve below (computed on validation) shows the optimum; the confusion
matrices show what the shift does on the test set.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "threshold_cost_curve.png")))

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "confusion_matrices.png")))

## ROC vs Precision-Recall

ROC-AUC looks near-perfect — with 56,651 genuine vs 95 fraud test rows, the false-positive *rate*
stays microscopic no matter what. The PR curve tells the honest story: precision visibly trades off
against recall. This contrast is the canonical argument for PR-AUC under extreme imbalance.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "roc_curve.png")))

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "pr_curve.png")))

## Business cost analysis

In [ ]:
bc = ev["business_cost"]
pd.Series(bc).to_frame("value")

## Explainability

**Permutation importance** (drop in PR-AUC when a feature is shuffled) is model-agnostic evidence of
what the model actually uses. **SHAP** attributes each individual prediction to features; the beeswarm
shows direction — e.g. strongly negative V14 pushes predictions toward fraud. The top features
(V14, V10, V12, V4, V17) are exactly the ones EDA flagged as most class-separating — a consistency
check across the whole project.

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "permutation_importance.png")))

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "shap_summary.png")))

In [ ]:
display(Image(str(resolve_path(config["paths"]["images_dir"]) / "feature_importance.png")))

## Inference smoke test

The deployable API: raw transaction in → probability, class, risk level out.

In [ ]:
from src.inference import FraudDetector
from src.data_loader import load_raw_data

detector = FraudDetector.load()
df = load_raw_data(config)
fraud_txn = df[df.Class == 1].iloc[0].drop("Class").to_dict()
genuine_txn = df[df.Class == 0].iloc[0].drop("Class").to_dict()
print("known fraud   ->", detector.predict_one(fraud_txn))
print("known genuine ->", detector.predict_one(genuine_txn))